# Accuracy of IDF evacuation orders in Gaza

In [21]:
!pip install deep-translator langchain langchain-google-genai

  Using cached langchain-0.3.25-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_google_genai-2.1.4-py3-none-any.whl.metadata (5.2 kB)
  Using cached langchain_text_splitters-0.3.8-py3-none-any.whl.metadata (1.9 kB)
  Using cached langsmith-0.3.42-py3-none-any.whl.metadata (15 kB)
     ---------------------------------------- 0.0/67.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/67.2 kB ? eta -:--:--
     ------ --------------------------------- 10.2/67.2 kB ? eta -:--:--
     ------ --------------------------------- 10.2/67.2 kB ? eta -:--:--
     ----------------------- -------------- 41.0/67.2 kB 393.8 kB/s eta 0:00:01
     -------------------------------------- 67.2/67.2 kB 518.0 kB/s eta 0:00:00
  Using cached sqlalchemy-2.0.41-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-win_amd64.whl.metadata (2.1 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_ai_generativelang


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# Install dependencies
import os
import time
import asyncio
from dotenv import load_dotenv
import re
import pandas as pd
from deep_translator import GoogleTranslator
from datetime import datetime, timedelta
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

## 1. Displacements

The [Gaza Maps](https://gazamaps.com/) project maintains a database of all known IDF evacuation orders issued on official IDF Arabic channels, or via leaflets in Gaza. 

In [23]:
# Load the displacement data
displacement = pd.read_csv("data/displacement_enriched.csv")
displacement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     109 non-null    object 
 1   x_source                 93 non-null     object 
 2   x_text                   93 non-null     object 
 3   x_text_translated        93 non-null     object 
 4   x_timestamp_utc3         93 non-null     object 
 5   facebook_source          21 non-null     object 
 6   facebook_timestamp_utc3  21 non-null     object 
 7   leaflet                  19 non-null     object 
 8   leaflet_text_translated  18 non-null     object 
 9   link                     95 non-null     object 
 10  map_idf                  95 non-null     object 
 11  map_full                 79 non-null     object 
 12  map_zoom                 79 non-null     object 
 13  displacement_blocks      85 non-null     object 
 14  labeled_safe_blocks      3

### Data cleaning
1. Remove the URL at the end of every tweet in the column `x_text`.
2. Translate every tweet from Arabic to English, creating a new column `x_text_translated`.
3. Convert the column `x_timestamp` to UTC+3, which is the timezone in Gaza.

In [24]:
def remove_urls(text):
    """Remove all URLs from the text."""
    if pd.isna(text) or not isinstance(text, str):
        return text # Leave non-string types unchanged
    return re.sub(r'https://\S+', '', text).strip()
    

def arabic_to_english(text):
    """Translate Arabic text to English."""
    if pd.isna(text) or not isinstance(text, str):
        return text
    try:
        return GoogleTranslator(source='ar', target='en').translate(text)
    except Exception as e:
        print(f"Error translating text: {text} - {e}")
        return text


def convert_date(time_str):
    """Convert a Twitter-style timestamp to hour:minute in UTC+3."""
    if pd.isna(time_str) or not isinstance(time_str, str):
        return time_str
    try:
        # Parse the original UTC datetime string
        dt = datetime.strptime(time_str, "%a %b %d %H:%M:%S %z %Y")     
        # Add 3 hours to get UTC+3
        dt_utc3 = dt + timedelta(hours=3)       
        # Format hour and minute
        time_formatted = dt_utc3.strftime("%H:%M")
        return time_formatted
    except Exception as e:
        print(f"Skipping malformed timestamp '{time_str}': {e}")
        return time_str

In [25]:
# Remove URLs from the tweet text
displacement['x_text'] = displacement['x_text'].apply(remove_urls)

# Translate Arabic tweets to English
# displacement['x_text_translated'] = displacement['x_text'].apply(arabic_to_english)

# Convert the timestamp to UTC+3 hour:minute format
# displacement['x_timestamp_utc3'] = displacement['x_timestamp'].apply(convert_date)

displacement.head()

,date,x_source,x_text,x_text_translated,x_timestamp_utc3,facebook_source,facebook_timestamp_utc3,leaflet,leaflet_text_translated,link,map_idf,map_full,map_zoom,displacement_blocks,labeled_safe_blocks,area_sq_km_displacement,area_sq_km_labeled_safe
0,2025-05-26,https://x.com/AvichayAdraee/status/19269564320...,#عاجل ‼️ الى سكان محافظة خانيونس، بني سهيلا، ع...,"To the residents of Khan Yunis, Bani Suhaila, ...",13:59,https://www.facebook.com/IDFarabicAvichayAdrae...,14:03,NaN,NaN,https://gazamaps.com/displacement/103,https://gazamaps.com/storage/displacement-maps...,NaN,NaN,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",NaN,154.44,0.0
1,2025-05-21,https://x.com/AvichayAdraee/status/19252451965...,#عاجل ‼️ تحذير خطير الى كل سكان قطاع غزة المتو...,A serious warning to all residents of the Gaza...,20:39,https://www.facebook.com/IDFarabicAvichayAdrae...,20:41,NaN,NaN,https://gazamaps.com/displacement/101,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"961, 962, 963, 964, 965, 966, 967, 968, 969, 9...",NaN,13.26,0.0
2,2025-05-19,https://x.com/AvichayAdraee/status/19243953249...,#عاجل ‼️ الى سكان محافظة خان يونس، بني سهيلا و...,"To the residents of Khan Yunis, Bani Suhaila a...",12:22,https://www.facebook.com/IDFarabicAvichayAdrae...,12:23,NaN,NaN,https://gazamaps.com/displacement/100,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"36, 37.1, 37.2, 38, 39, 40, 41, 42.1, 42.2, 42...",NaN,81.24,0.0
3,2025-05-18,https://x.com/avichayadraee/status/19241385814...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في...,To all residents of the Gaza Strip located in ...,19:22,https://www.facebook.com/IDFarabicAvichayAdrae...,19:23,NaN,NaN,https://gazamaps.com/displacement/99,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"126, 127, 129.1, 129.2, 130.1, 130.2, 131.1, 2...",NaN,5.71,0.0
4,2025-05-16,NaN,NaN,NaN,NaN,NaN,NaN,https://idfleaflets.com/53,URGENT WARNING:\n\nTo all those who are in thi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The tweet text `x_text_translated` contains the names of the locations designated to be evacuated, and the locations where residents should find shelter. We will use an LLM to extract these locations and put them in separate columns for further analysis.

> Note: Method currently not working. We used the SheetGPT extension for Google Sheets instead with the same prompts.

In [ ]:
try:
    # Load Google Gemini API key
    dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
    load_dotenv(dotenv_path)

    google_api_key = os.getenv("GOOGLE_API_KEY")
except:
    google_api_key = input("Your Google API key: ")

# Set up the LLM
llm = GoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=google_api_key)

# Global delay tracker
last_call_time = 0

async def throttled_llm_run(chain, input_data, delay=4):
    """Run LLM chain with rate limit delay (default: 4 seconds)."""
    global last_call_time

    now = time.time()
    time_since_last = now -last_call_time

    if time_since_last < delay:
        await asyncio.sleep(delay - time_since_last)
    
    try:
        result = chain.run(x_text_translated=input_data).strip()
        last_call_time = time.time()  # Update the last call time
        return result
    except Exception as e:
        print(f"Error running LLM chain: {e}")
        return None

In [ ]:
async def extract_evacuations(row):
    """Function to extract the names of locations designated to be evacuated from a tweet."""
    prompt = PromptTemplate.from_template("""
    You will be given a tweet containing an evacuation order to residents of specific locations.
    Name the affected locations (not the block numbers, if present in the tweet) where residents are ordered to evacuate from, separated by commas.
    If the location names where residents should evacuate from are not mentioned in the tweet, answer "Unknown".
                                          
    Tweet: {x_text_translated}
    """)

    chain = LLMChain(llm=llm, prompt=prompt)

    if isinstance(row['x_text_translated'], str):
        # Prompt the LLM to extract evacuation locations
        return await throttled_llm_run(chain, row['x_text_translated'])
    return None


async def extract_safezones(row):
    """Function to extract the names of safe zones from a tweet."""
    prompt = PromptTemplate.from_template("""
    You will be given a tweet containing an evacuation order to residents of specific locations.
    Name the safe zones (not the block numbers, if present in the tweet) where residents are ordered to evacuate to, separated by commas.
    If the safe zones where residents should evacuate to are not mentioned in the tweet, answer "Unknown".
                                          
    Tweet: {x_text_translated}
    """)

    chain = LLMChain(llm=llm, prompt=prompt)

    if isinstance(row['x_text_translated'], str):
        return await throttled_llm_run(chain, row['x_text_translated'])
    return None

In [ ]:
# Extract evacuation locations from the translated tweets
# displacement["locations"] = displacement.apply(extract_evacuations, axis=1)

# Extract safe zones from the translated tweets
# displacement["safezones"] = displacement.apply(extract_safezones, axis=1)

## 2. Political violence events

ACLED's [Gaza Monitor](https://acleddata.com/gaza-monitor/#1738833681742-00591c0f-83dc) contains a map and accompanying dataset of all political violence events, demonstration events, and strategic developments recorded in Israel and Palestine started from 7 October 2023.

In [19]:
# Load the ACLED data
acled = pd.read_csv("data/acled_may23.csv")
acled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90242 entries, 0 to 90241
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   event_id_cnty       90242 non-null  object 
 1   event_date          90242 non-null  object 
 2   year                90242 non-null  int64  
 3   time_precision      90242 non-null  int64  
 4   disorder_type       90242 non-null  object 
 5   event_type          90242 non-null  object 
 6   sub_event_type      90242 non-null  object 
 7   actor1              90242 non-null  object 
 8   assoc_actor_1       20743 non-null  object 
 9   inter1              90242 non-null  object 
 10  actor2              66166 non-null  object 
 11  assoc_actor_2       16778 non-null  object 
 12  inter2              66166 non-null  object 
 13  interaction         90242 non-null  object 
 14  civilian_targeting  21230 non-null  object 
 15  iso                 90242 non-null  int64  
 16  regi

### Data cleaning

The [ACLED Codebook](https://acleddata.com/acleddatanew/wp-content/uploads/dlm_uploads/2024/10/ACLED-Codebook-2024-7-Oct.-2024.pdf) contains more information about how ACLED codes and catagorizes data. Based on this we will filter the dataset to only include battles, explosions/remote violence and violence against civilians by the IDF in Gaza after 7 October 2023:

1. Only include dates in the column `event_date` after 2023-10-07.
2. Only include datapoints for 'Political violence' in the column `disorder_type`. Only include the following subcategories within the column `event_type`:
    - Battles
    - Explosions/Remote Violence
    - Violence against civilians
3. Only include datapoints that contain 'Military Forces of Israel' in the column `actor1`.
4. Only include datapoints for the location 'Gaza Strip' in the column `admin1`.

In [ ]:
# Ensure event_date is in datetime format
acled["event_date"] = pd.to_datetime(acled["event_date"], errors='coerce')

# Filter events types
event_types = ["Battles", "Explosions/Remote Violence", "Violence against civilians"]

# Apply filters
acled_filtered = acled[
    (acled["event_date"] > pd.Timestamp("2023-10-07")) &
    (acled["disorder_type"] == "Political violence") &
    (acled["event_type"].isin(event_types)) &
    (acled["actor1"].str.contains("Military Forces of Israel", na=False)) &
    (acled["admin1"] == "Gaza Strip")
]

acled_filtered.head()